# A1 Tensor Product Exploration

Question to keep in mind:

- If I tensor-product two features with the same irreps, does the output stay the same?

Short answer:

- Not automatically.
- A full tensor product expands into every allowed coupling channel.
- You only stay in the same representation type if you explicitly choose `irreps_out` to be that same irrep set.
- Even then, the output values are new bilinear features, not a copy of the input.

This notebook uses the repo's `A1` embedding definitions from `models/local_iso_embedding.py` and the same `FullyConnectedTensorProduct` style used in `models/SR_ocrp.py`.

In [1]:
from collections import Counter
from pathlib import Path
import sys

import torch
from e3nn import o3

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "models").exists():
    repo_root = repo_root.parent
if not (repo_root / "models").exists():
    raise RuntimeError("Could not locate repo root containing a models/ directory")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from models.local_iso_embedding import (
    build_local_iso_fcc_embedding,
    build_local_iso_hcp_embedding,
)
from models.SR_ocrp import CosineMaskedEquivariantSpatialConv, LocalIsoCrystalEncoder

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

## 1. Repo-specific `A1` irreps

In this repo, `irreps_a1` does **not** mean `0e` scalars only.

- `models/local_iso_embedding.py` first computes how many copies of crystal-group `A1` live inside each SO(3) `l` band.
- It then prunes away inactive bands and builds the active feature space `irreps_a1`.

So the model's `A1` feature space is a symmetry-filtered subspace of nonscalar SO(3) irreps.

In [2]:
fcc = build_local_iso_fcc_embedding(device="cpu")
hcp = build_local_iso_hcp_embedding(device="cpu")

print("FCC irreps_a1:", fcc.irreps_a1, "dim=", fcc.irreps_a1.dim)
print("HCP irreps_a1:", hcp.irreps_a1, "dim=", hcp.irreps_a1.dim)
print("FCC irreps_full:", fcc.irreps_full)
print("HCP irreps_full:", hcp.irreps_full)

FCC irreps_a1: 1x4e dim= 9
HCP irreps_a1: 2x2e+1x4e+1x6e dim= 32
FCC irreps_full: 1x2e+1x4e
HCP irreps_full: 2x2e+1x4e+1x6e


In [3]:
def describe_self_product(name, irreps):
    irreps = o3.Irreps(irreps)
    full_tp = o3.FullTensorProduct(irreps, irreps)
    same_tp = o3.FullyConnectedTensorProduct(irreps, irreps, irreps, shared_weights=True)

    print(f"=== {name} ===")
    print("in      :", irreps)
    print("dim     :", irreps.dim)
    print("full out:", full_tp.irreps_out)
    print("full dim:", full_tp.irreps_out.dim)
    print("same out:", same_tp.irreps_out)
    print("weights :", same_tp.weight_numel)
    print()


def count_paths_by_output_irrep(tp):
    counts = Counter()
    for inst in tp.instructions:
        counts[inst.i_out] += 1
    rows = []
    for i_out, mul_ir in enumerate(tp.irreps_out):
        rows.append((i_out, str(mul_ir), counts[i_out]))
    return rows

In [4]:
describe_self_product("scalar 0e", "1x0e")
describe_self_product("pure 4e", "1x4e")
describe_self_product("repo FCC A1", fcc.irreps_a1)
describe_self_product("repo HCP A1", hcp.irreps_a1)

=== scalar 0e ===
in      : 1x0e
dim     : 1
full out: 1x0e
full dim: 1
same out: 1x0e
weights : 1

=== pure 4e ===
in      : 1x4e
dim     : 9
full out: 1x0e+1x1e+1x2e+1x3e+1x4e+1x5e+1x6e+1x7e+1x8e
full dim: 81
same out: 1x4e
weights : 1

=== repo FCC A1 ===
in      : 1x4e
dim     : 9
full out: 1x0e+1x1e+1x2e+1x3e+1x4e+1x5e+1x6e+1x7e+1x8e
full dim: 81
same out: 1x4e
weights : 1

=== repo HCP A1 ===
in      : 2x2e+1x4e+1x6e
dim     : 32
full out: 4x0e+1x0e+1x0e+4x1e+1x1e+1x1e+4x2e+2x2e+2x2e+1x2e+1x2e+1x2e+1x2e+4x3e+2x3e+2x3e+1x3e+1x3e+1x3e+1x3e+4x4e+2x4e+2x4e+2x4e+1x4e+1x4e+2x4e+1x4e+1x4e+2x5e+2x5e+2x5e+1x5e+1x5e+2x5e+1x5e+1x5e+2x6e+2x6e+2x6e+1x6e+1x6e+2x6e+1x6e+1x6e+2x7e+1x7e+1x7e+2x7e+1x7e+1x7e+2x8e+1x8e+1x8e+2x8e+1x8e+1x8e+1x9e+1x9e+1x9e+1x10e+1x10e+1x10e+1x11e+1x12e
full dim: 1024
same out: 2x2e+1x4e+1x6e
weights : 52



Interpretation:

- `0e x 0e -> 0e`, so the scalar case really does stay scalar even for the full product.
- `4e x 4e` does **not** stay `4e` by default. The full product expands to `0e + 1e + ... + 8e`.
- But `FullyConnectedTensorProduct(4e, 4e, 4e)` is valid because the `4e` channel is one allowed part of that decomposition.
- For repo HCP `A1`, self-product has many available couplings, and projecting back to the same `A1` space still leaves a rich learned bilinear map.

In [5]:
irr = o3.Irreps("1x4e")
full_tp = o3.FullTensorProduct(irr, irr)
same_tp = o3.FullyConnectedTensorProduct(irr, irr, irr, shared_weights=True)

x = torch.randn(3, irr.dim)
out_full = full_tp(x, x)
out_same = same_tp(x, x)

print("x shape       :", tuple(x.shape))
print("full out shape:", tuple(out_full.shape), "irreps_out=", full_tp.irreps_out)
print("same out shape:", tuple(out_same.shape), "irreps_out=", same_tp.irreps_out)
print("mean ||out_same - x||:", (out_same - x).norm(dim=-1).mean().item())
print("max |tp(2x, x) - 2 tp(x, x)|:", (same_tp(2 * x, x) - 2 * out_same).abs().max().item())
print("max |tp(2x, 2x) - 4 tp(x, x)|:", (same_tp(2 * x, 2 * x) - 4 * out_same).abs().max().item())

x shape       : (3, 9)
full out shape: (3, 81) irreps_out= 1x0e+1x1e+1x2e+1x3e+1x4e+1x5e+1x6e+1x7e+1x8e
same out shape: (3, 9) irreps_out= 1x4e
mean ||out_same - x||: 2.4271438121795654
max |tp(2x, x) - 2 tp(x, x)|: 0.0
max |tp(2x, 2x) - 4 tp(x, x)|: 0.0


This cell makes the key distinction explicit:

- `same_tp` returns a feature that **transforms as** `4e`.
- It does **not** return the original `x`.
- It is a learned bilinear coupling of the two inputs.

So “same irreps out” means same representation type, not identity or passthrough.

In [6]:
hcp_same_tp = o3.FullyConnectedTensorProduct(
    hcp.irreps_a1,
    hcp.irreps_a1,
    hcp.irreps_a1,
    shared_weights=True,
)

print("HCP A1 same-out weight_numel:", hcp_same_tp.weight_numel)
print("HCP A1 path counts by output block:")
for row in count_paths_by_output_irrep(hcp_same_tp):
    print(row)

HCP A1 same-out weight_numel: 52
HCP A1 path counts by output block:
(0, '2x2e', 7)
(1, '1x4e', 9)
(2, '1x6e', 8)


In [7]:
fcc_conv = CosineMaskedEquivariantSpatialConv(
    kernel_size=3,
    irreps_in=fcc.irreps_a1,
    irreps_out=fcc.irreps_a1,
    use_residual=False,
)

hcp_conv = CosineMaskedEquivariantSpatialConv(
    kernel_size=3,
    irreps_in=hcp.irreps_a1,
    irreps_out=hcp.irreps_a1,
    use_residual=False,
)

print("FCC conv tp:")
print("  in1:", fcc_conv.tp.irreps_in1)
print("  in2:", fcc_conv.tp.irreps_in2)
print("  out:", fcc_conv.tp.irreps_out)
print("  weight_numel:", fcc_conv.tp.weight_numel)
print()
print("HCP conv tp:")
print("  in1:", hcp_conv.tp.irreps_in1)
print("  in2:", hcp_conv.tp.irreps_in2)
print("  out:", hcp_conv.tp.irreps_out)
print("  weight_numel:", hcp_conv.tp.weight_numel)

FCC conv tp:
  in1: 1x4e
  in2: 1x4e
  out: 1x4e
  weight_numel: 1

HCP conv tp:
  in1: 2x2e+1x4e+1x6e
  in2: 2x2e+1x4e+1x6e
  out: 2x2e+1x4e+1x6e
  weight_numel: 52


## Bottom line

- If you TP two inputs with the same irreps, the output does **not** automatically remain in that same irrep set.
- The unconstrained full product usually expands into many irreps.
- In this repo, layers stay in `irreps_a1` because we explicitly construct `FullyConnectedTensorProduct(irreps_a1, irreps_a1, irreps_a1)`.
- That preserves the **feature type** seen by later equivariant layers, but it still computes a new learned bilinear feature.
- For FCC `A1 = 1x4e`, that same-out map is especially narrow: one coupling path.
- For HCP `A1 = 2x2e+1x4e+1x6e`, same-out TP is richer: many coupling paths back into the same `A1` space.

Useful next experiment:

- Compare `FullyConnectedTensorProduct(irreps_a1, irreps_a1, irreps_a1)` against a wider `irreps_out` and see which added channels help downstream decoding.

## FCC `a1` embedding self-product on a real quaternion

Question:

- Take a random quaternion.
- Compute the repo's FCC `a1` embedding.
- Apply `FullyConnectedTensorProduct(irreps_a1, irreps_a1, irreps_a1)` to that embedding with itself.

What should happen?

- You should **not** expect to get the exact same feature back.
- The layer is a **learned bilinear map**, not an identity map.
- For FCC `A1 = 1x4e`, there is only one same-out coupling path, so on the rotation-orbit generated by the embedding the self-product can become proportional to the input direction.
- That is a special property of this restricted orbit, not a generic fact about arbitrary `4e` vectors.


In [8]:
torch.manual_seed(7)

encoder = LocalIsoCrystalEncoder(crystal="fcc", device="cpu")
q = torch.randn(1, 4)
q = q / q.norm(dim=-1, keepdim=True)
x = encoder.forward_a1(q)

fcc_same_tp = o3.FullyConnectedTensorProduct(
    encoder.irreps_a1,
    encoder.irreps_a1,
    encoder.irreps_a1,
    shared_weights=True,
)
fcc_full_tp = o3.FullTensorProduct(encoder.irreps_a1, encoder.irreps_a1)

y_same = fcc_same_tp(x, x)
y_full = fcc_full_tp(x, x)
c = ((y_same * x).sum() / (x * x).sum()).item()

print("q:", q)
print("FCC irreps_a1:", encoder.irreps_a1, "dim=", encoder.irreps_a1.dim)
print("same_tp weight_numel:", fcc_same_tp.weight_numel)
print("full_tp irreps_out:", fcc_full_tp.irreps_out)
print("x shape:", tuple(x.shape))
print("y_same shape:", tuple(y_same.shape))
print("mean ||y_same - x||:", (y_same - x).norm(dim=-1).mean().item())
print("cos(y_same, x):", torch.nn.functional.cosine_similarity(y_same, x, dim=-1))
print("best proportional scale c:", c)
print("max |y_same - c x|:", (y_same - c * x).abs().max().item())
print("max |tp(2x, x) - 2 tp(x, x)|:", (fcc_same_tp(2 * x, x) - 2 * y_same).abs().max().item())
print("max |tp(2x, 2x) - 4 tp(x, x)|:", (fcc_same_tp(2 * x, 2 * x) - 4 * y_same).abs().max().item())

w0 = fcc_same_tp.weight.detach().clone()
fcc_same_tp.weight.data.fill_(0.0)
y_zero = fcc_same_tp(x, x)
fcc_same_tp.weight.data.fill_(1.0)
y_one = fcc_same_tp(x, x)
fcc_same_tp.weight.data.fill_(2.0)
y_two = fcc_same_tp(x, x)
fcc_same_tp.weight.data.copy_(w0)

print("max |y_zero|:", y_zero.abs().max().item())
print("max |y_two - 2 y_one|:", (y_two - 2 * y_one).abs().max().item())

x_rand = torch.randn(5, encoder.irreps_a1.dim)
y_rand = fcc_same_tp(x_rand, x_rand)
print("generic random 4e cosines:", torch.nn.functional.cosine_similarity(y_rand, x_rand, dim=-1))


q: tensor([[-0.0881,  0.4717,  0.5681, -0.6686]])
FCC irreps_a1: 1x4e dim= 9
same_tp weight_numel: 1
full_tp irreps_out: 1x0e+1x1e+1x2e+1x3e+1x4e+1x5e+1x6e+1x7e+1x8e
x shape: (1, 9)
y_same shape: (1, 9)
mean ||y_same - x||: 0.26608097553253174
cos(y_same, x): tensor([1.], grad_fn=<SumBackward1>)
best proportional scale c: 0.31298255920410156
max |y_same - c x|: 4.0978193283081055e-08
max |tp(2x, x) - 2 tp(x, x)|: 0.0
max |tp(2x, 2x) - 4 tp(x, x)|: 0.0
max |y_zero|: 0.0
max |y_two - 2 y_one|: 0.0
generic random 4e cosines: tensor([ 0.0295, -0.5361,  0.5569, -0.0457, -0.5134], grad_fn=<SumBackward1>)


## Why the FCC `a1` self-product is proportional on the quaternion orbit

Let `V = 1x4e`, the single SO(3) `l=4` block used by repo FCC `a1`.

Write the FCC `a1` embedding of an orientation `g` as

- `x(g) = D^(4)(g) v_A1`

where `v_A1` is the cubic-`A1` basis vector at the reference orientation and `D^(4)` is the `l=4` representation matrix.

Now define the self-product map

- `y(x) = TP(x, x)`

This map is **equivariant**, so

- `y(D(g) x) = D(g) y(x)`

for the `same-out` tensor product.

Evaluate this at the reference vector `v_A1`.
For every cubic symmetry `h in O`, the reference `A1` vector is fixed:

- `D^(4)(h) v_A1 = v_A1`

By equivariance,

- `D^(4)(h) y(v_A1) = y(D^(4)(h) v_A1) = y(v_A1)`

So `y(v_A1)` is also fixed by the cubic subgroup.
Inside a single `l=4` block, the cubic-invariant subspace is exactly the 1D `A1` line, so `y(v_A1)` must lie on that same line:

- `y(v_A1) = c v_A1`

for some scalar `c` that depends on the TP weights.

Then for any orientation `g`,

- `y(x(g)) = y(D^(4)(g) v_A1) = D^(4)(g) y(v_A1) = c D^(4)(g) v_A1 = c x(g)`

So on the **rotation orbit generated by the FCC `a1` embedding**, the self-TP is forced by symmetry to be proportional to the input direction.

Important caveat:

- This does **not** hold for arbitrary random vectors in `1x4e`.
- It holds on the special orbit `x(g) = D^(4)(g) v_A1` coming from the repo embedding.
- Also note that this argument uses the stabilizer of the reference `A1` vector, not Schur's lemma directly, because `x -> TP(x, x)` is nonlinear.


In [9]:
torch.manual_seed(11)

encoder = LocalIsoCrystalEncoder(crystal="fcc", device="cpu")
tp_orbit = o3.FullyConnectedTensorProduct(
    encoder.irreps_a1,
    encoder.irreps_a1,
    encoder.irreps_a1,
    shared_weights=True,
)
tp_orbit.weight.data.fill_(1.0)

scales = []
residuals = []
for _ in range(32):
    q = torch.randn(1, 4)
    q = q / q.norm(dim=-1, keepdim=True)
    x = encoder.forward_a1(q)
    y = tp_orbit(x, x)
    c = ((y * x).sum() / (x * x).sum()).item()
    scales.append(c)
    residuals.append((y - c * x).abs().max().item())

scales = torch.tensor(scales)
residuals = torch.tensor(residuals)
print("orbit scale min/max/std:", scales.min().item(), scales.max().item(), scales.std().item())
print("orbit residual max:", residuals.max().item())

x_rand = torch.randn(32, encoder.irreps_a1.dim)
y_rand = tp_orbit(x_rand, x_rand)
cos_rand = torch.nn.functional.cosine_similarity(y_rand, x_rand, dim=-1)
print("generic random 4e cosine min/max/mean:", cos_rand.min().item(), cos_rand.max().item(), cos_rand.mean().item())


orbit scale min/max/std: 0.18511003255844116 0.18511027097702026 5.6139121795695246e-08
orbit residual max: 2.7939677238464355e-08
generic random 4e cosine min/max/mean: -0.8647164702415466 0.9413970112800598 -0.11818718165159225


## Learn the same-out FCC `a1` TP weight to reproduce the input

For FCC repo `a1`, the same-out TP has exactly one learned scalar weight.

Because on the quaternion orbit we observed

- `TP_w=1(x, x) = c x`

the best weight for reconstructing the input should be close to

- `w* = 1 / c`

This cell optimizes that single TP weight directly and compares the learned value to the analytic prediction.


In [10]:
torch.manual_seed(123)

encoder = LocalIsoCrystalEncoder(crystal="fcc", device="cpu")

# Estimate the proportionality constant c for weight=1 on the quaternion orbit.
tp_ref = o3.FullyConnectedTensorProduct(
    encoder.irreps_a1,
    encoder.irreps_a1,
    encoder.irreps_a1,
    shared_weights=True,
)
tp_ref.weight.data.fill_(1.0)

q_probe = torch.randn(128, 4)
q_probe = q_probe / q_probe.norm(dim=-1, keepdim=True)
x_probe = encoder.forward_a1(q_probe)
y_probe = tp_ref(x_probe, x_probe)
c_hat = ((y_probe * x_probe).sum(dim=-1) / (x_probe * x_probe).sum(dim=-1)).mean().item()
w_star_pred = 1.0 / c_hat

# Learn the single weight by minimizing ||TP(x,x) - x||^2.
tp_learn = o3.FullyConnectedTensorProduct(
    encoder.irreps_a1,
    encoder.irreps_a1,
    encoder.irreps_a1,
    shared_weights=True,
)
tp_learn.weight.data.fill_(0.0)

opt = torch.optim.Adam([tp_learn.weight], lr=0.5)
history = []
for step in range(400):
    q = torch.randn(256, 4)
    q = q / q.norm(dim=-1, keepdim=True)
    x = encoder.forward_a1(q)
    y = tp_learn(x, x)
    loss = ((y - x) ** 2).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
    history.append((step, tp_learn.weight.item(), loss.item()))

print("same-out irreps:", tp_learn.irreps_out)
print("weight_numel:", tp_learn.weight_numel)
print("predicted optimal weight 1/c:", w_star_pred)
print("learned weight:", tp_learn.weight.item())
print("abs error in learned weight:", abs(tp_learn.weight.item() - w_star_pred))
print("final loss:", history[-1][2])
print()
print("last 5 optimization steps:")
for row in history[-5:]:
    print(row)

# Sanity check on a fresh batch.
q_test = torch.randn(64, 4)
q_test = q_test / q_test.norm(dim=-1, keepdim=True)
x_test = encoder.forward_a1(q_test)
y_test = tp_learn(x_test, x_test)
print()
print("test mean ||TP(x,x) - x||:", (y_test - x_test).norm(dim=-1).mean().item())
print("test mean cosine:", torch.nn.functional.cosine_similarity(y_test, x_test, dim=-1).mean().item())


same-out irreps: 1x4e
weight_numel: 1
predicted optimal weight 1/c: 5.402189731077325
learned weight: 5.402189254760742
abs error in learned weight: 4.7631658262048404e-07
final loss: 2.3985315500369775e-15

last 5 optimization steps:
(395, 5.402189254760742, 2.6145980081785248e-15)
(396, 5.402189254760742, 2.333403821997305e-15)
(397, 5.402189254760742, 2.4583634163321795e-15)
(398, 5.402189254760742, 2.3605783330111173e-15)
(399, 5.402189254760742, 2.3985315500369775e-15)

test mean ||TP(x,x) - x||: 1.4154718996906013e-07
test mean cosine: 1.0


## Target `TP(x, weighted-sum)` to reproduce `x`

Now use:

- a target embedding `x = phi(q0)` from one random quaternion
- a context set containing `x` itself plus several distractor quaternion embeddings
- a softmax-weighted context sum `s = sum_i a_i x_i`
- a learnable same-out TP `y = TP(x, s)`

Target:

- make `y` equal to `x`

What should the network learn?

- For FCC repo `a1`, `TP(x, x)` is already proportional to `x`.
- So the easiest and most robust solution is:
  1. put almost all context weight on the `x` term itself
  2. let the TP's single learned scalar weight absorb the required scale factor

In other words, with a normalized attention-style sum, the natural expectation is:

- `a_self -> 1`
- distractor weights `-> 0`
- TP weight `-> 1 / c`

where `c` is the proportionality constant in `TP_w=1(x, x) = c x`.

Important caveat:

- If you use unconstrained context coefficients and also learn the TP weight, the problem has scale non-identifiability and many exact solutions.
- The softmax context sum makes the interpretation cleaner: selection happens in the context weights, scale happens in the TP weight.


In [11]:
torch.manual_seed(2026)

encoder = LocalIsoCrystalEncoder(crystal="fcc", device="cpu")

# Build one target quaternion plus several distractors.
q0 = torch.randn(1, 4)
q0 = q0 / q0.norm(dim=-1, keepdim=True)
q_distr = torch.randn(4, 4)
q_distr = q_distr / q_distr.norm(dim=-1, keepdim=True)
q_ctx = torch.cat([q0, q_distr], dim=0)
x_ctx = encoder.forward_a1(q_ctx)
x = x_ctx[:1]   # target feature

# Estimate c from TP_w=1(x, x) = c x.
tp_ref = o3.FullyConnectedTensorProduct(
    encoder.irreps_a1,
    encoder.irreps_a1,
    encoder.irreps_a1,
    shared_weights=True,
)
tp_ref.weight.data.fill_(1.0)
c_hat = ((tp_ref(x, x) * x).sum() / (x * x).sum()).item()
w_star_pred = 1.0 / c_hat

# Learn both the context weights and the same-out TP weight.
tp_mix = o3.FullyConnectedTensorProduct(
    encoder.irreps_a1,
    encoder.irreps_a1,
    encoder.irreps_a1,
    shared_weights=True,
)
tp_mix.weight.data.fill_(0.0)
logits = torch.nn.Parameter(torch.zeros(x_ctx.shape[0]))
opt = torch.optim.Adam([tp_mix.weight, logits], lr=0.1)

history = []
for step in range(800):
    attn = torch.softmax(logits, dim=0)
    s = (attn.view(-1, 1) * x_ctx).sum(dim=0, keepdim=True)
    y = tp_mix(x, s)
    loss = ((y - x) ** 2).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 100 == 0 or step == 799:
        history.append((step, tp_mix.weight.item(), attn.detach().clone(), loss.item()))

attn = torch.softmax(logits, dim=0)
s = (attn.view(-1, 1) * x_ctx).sum(dim=0, keepdim=True)
y = tp_mix(x, s)

print("predicted TP weight 1/c:", w_star_pred)
print("learned TP weight:", tp_mix.weight.item())
print("learned attention weights:", attn)
print("argmax context index (0 means self):", int(attn.argmax().item()))
print("mean ||TP(x, s) - x||:", (y - x).norm(dim=-1).mean().item())
print("cos(TP(x, s), x):", torch.nn.functional.cosine_similarity(y, x, dim=-1).item())
print()
print("optimization snapshots:")
for step, w_val, a_val, loss_val in history:
    print(step, w_val, a_val, loss_val)


predicted TP weight 1/c: 5.4021871218566
learned TP weight: 5.424917221069336
learned attention weights: tensor([    0.9955,     0.0007,     0.0025,     0.0006,     0.0006],
       grad_fn=<SoftmaxBackward0>)
argmax context index (0 means self): 0
mean ||TP(x, s) - x||: 0.0004254991945344955
cos(TP(x, s), x): 0.9999994039535522

optimization snapshots:
0 0.0999981015920639 tensor([0.2000, 0.2000, 0.2000, 0.2000, 0.2000]) 0.01666668988764286
100 5.460490703582764 tensor([    0.9954,     0.0007,     0.0026,     0.0006,     0.0007]) 9.311598887506989e-07
200 5.425192356109619 tensor([    0.9954,     0.0007,     0.0026,     0.0006,     0.0007]) 2.182644820436508e-08
300 5.425400733947754 tensor([    0.9954,     0.0007,     0.0026,     0.0006,     0.0007]) 2.1582165388167596e-08
400 5.425321578979492 tensor([    0.9954,     0.0007,     0.0026,     0.0006,     0.0007]) 2.1343616651847697e-08
500 5.425233840942383 tensor([    0.9955,     0.0007,     0.0026,     0.0006,     0.0007]) 2.10719228

## Add a `0.5 * input` residual-style term

Now change the map to

- `y = 0.5 * x + TP(x, s)`

with the same softmax context sum `s` over `{x, distractors}`.

If the model still chooses the self term, then with `TP_w=1(x, x) = c x` we expect

- `0.5 * x + w * c * x \approx x`

so the TP weight should move from `1 / c` down to roughly

- `w* = 0.5 / c`

This mirrors the idea of keeping half of the input as a residual anchor while asking the TP branch to supply the rest.


In [ ]:
torch.manual_seed(2027)

encoder = LocalIsoCrystalEncoder(crystal="fcc", device="cpu")
beta = 0.5

# Build one target quaternion plus several distractors.
q0 = torch.randn(1, 4)
q0 = q0 / q0.norm(dim=-1, keepdim=True)
q_distr = torch.randn(4, 4)
q_distr = q_distr / q_distr.norm(dim=-1, keepdim=True)
q_ctx = torch.cat([q0, q_distr], dim=0)
x_ctx = encoder.forward_a1(q_ctx)
x = x_ctx[:1]

# Estimate c from TP_w=1(x, x) = c x.
tp_ref = o3.FullyConnectedTensorProduct(
    encoder.irreps_a1,
    encoder.irreps_a1,
    encoder.irreps_a1,
    shared_weights=True,
)
tp_ref.weight.data.fill_(1.0)
c_hat = ((tp_ref(x, x) * x).sum() / (x * x).sum()).item()
w_star_pred = beta / c_hat

# Learn both the context weights and the same-out TP weight for y = beta*x + TP(x, s).
tp_mix = o3.FullyConnectedTensorProduct(
    encoder.irreps_a1,
    encoder.irreps_a1,
    encoder.irreps_a1,
    shared_weights=True,
)
tp_mix.weight.data.fill_(0.0)
logits = torch.nn.Parameter(torch.zeros(x_ctx.shape[0]))
opt = torch.optim.Adam([tp_mix.weight, logits], lr=0.1)

history = []
for step in range(800):
    attn = torch.softmax(logits, dim=0)
    s = (attn.view(-1, 1) * x_ctx).sum(dim=0, keepdim=True)
    y = beta * x + tp_mix(x, s)
    loss = ((y - x) ** 2).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 100 == 0 or step == 799:
        history.append((step, tp_mix.weight.item(), attn.detach().clone(), loss.item()))

attn = torch.softmax(logits, dim=0)
s = (attn.view(-1, 1) * x_ctx).sum(dim=0, keepdim=True)
y = beta * x + tp_mix(x, s)

print("predicted TP weight beta/c:", w_star_pred)
print("learned TP weight:", tp_mix.weight.item())
print("learned attention weights:", attn)
print("argmax context index (0 means self):", int(attn.argmax().item()))
print("mean ||0.5*x + TP(x, s) - x||:", (y - x).norm(dim=-1).mean().item())
print("cos(0.5*x + TP(x, s), x):", torch.nn.functional.cosine_similarity(y, x, dim=-1).item())
print()
print("optimization snapshots:")
for step, w_val, a_val, loss_val in history:
    print(step, w_val, a_val, loss_val)
